In [ ]:
import os
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from thefuzz import fuzz
from tqdm import tqdm
from func_timeout import func_timeout, FunctionTimedOut
import multiprocessing

nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
ner_df_path = r'C:\project\political_ner\Insta_TikTok_AccountNames\NER_Identify\Sweden_NER.xlsx'
df_ner = pd.read_excel(ner_df_path)

df_ner.drop(df_ner.columns[2], axis=1, inplace=True) 
df_ner.drop_duplicates(subset=[df_ner.columns[1]], keep='first', inplace=True) #remove duplicates in col 2
df_ner

In [ ]:
import openai
import pandas as pd
import json
from tqdm import tqdm
from requests.exceptions import Timeout

openai.api_key = os.environ["OPENAI_API_KEY"]

SYSTEM_PROMPT = (
    "You are a specialized Named Entity Recognition (NER) system focused exclusively "
    "on extracting names of persons, political parties,their abbreviations or account names. Your task is to identify "
    "all names of individuals, political parties, abbreviations or account names (instagram or tiktok user names starting with something like @ or having official etc) mentioned in a given sentence, "
    "particularly in the context of political actors like Members of the European "
    "Parliament (MEPs) and other notable persons. The country focus is SWEDEN, "
    "but it can include those across EU and USA as well. "
    "Your extraction should be precise and ensure full names are captured when available. "
    "Even if the spelling is incorrect, extract the names, parties, and abbreviations as they appear."
    "These are mostly seperated by comma as well. Account names include any possible account name like 'official', 'tv', '.something' etc."
    "Sometimes, these account may not make sense as well (they may seem abstract or jibberish) they can still be important, so please get them as well)"
)

USER_PROMPT = "Are you clear about your role?"

ASSISTANT_PROMPT = (
    "Yes, I understand. Please provide the text, and I will extract all the persons' "
    "names, political party names, and abbreviations as required."
)

def openai_chat_completion_response(sentence, timeout=10):
    """
    Sends a prompt to the OpenAI ChatCompletion API and requests a strict JSON response.
    If the response is invalid JSON, tries to handle it gracefully.
    """

    prompt_text = (
        "Entity Definitions:\n"
        "1) PERSON: Any mention of an individual's name (politicians, MEPs, etc.).\n"
        "2) PARTY: Any mention of political parties or organizations affiliated with individuals.\n"
        "3) ABBREVIATION: Any abbreviation related to political parties or individuals. For example PP, PS etc.\n\n"
        "3) ACCOUNT NAMES: Any thing which looks like a account name, get it in full. For example they start woth @ or have . in it or have 'offical' in it etc\n\n"
        "Output Format (valid JSON only):\n"
        "{\n"
        '  "NER": ["List", "Of", "PERSON, PARTY, ABBREVIATION, OR ACCOUNT NAME"],\n'
        "}\n\n"
        "Important: Respond ONLY with valid JSON. Do not include any extra text. No extra text or explaination. \n\n"
        f"Sentence to analyze: {sentence}\n"
    
    )

    try:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",  
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT},
                {"role": "assistant", "content": ASSISTANT_PROMPT},
                {"role": "user", "content": prompt_text},
            ],
            timeout=timeout,
            temperature=0
        )

        raw_text = response.choices[0].message.content.strip()

        parsed_json = json.loads(raw_text)  
        return parsed_json

    except Timeout:
        return {"NER": ["Error: Timeout"]}
    except json.JSONDecodeError:
        return {"NER": ["Error: Invalid JSON"]}
    except Exception as e:
        return {"NER": [f"Error: {str(e)}"]}

df_ner['NER'] = ""  

for idx, row in tqdm(df_ner.iterrows(), total=len(df_ner), desc="Processing NER"):
    text = row.get("original", "")
    if pd.isna(text):
        text = ""

    ner_result = openai_chat_completion_response(text)

    entities = ner_result.get("NER", [])
    combined_str = ", ".join(entities)

    df_ner.at[idx, "NER"] = combined_str



In [ ]:
df_ner

In [ ]:
#Saving to a seperate folder, a new clean folder 
df_ner.to_excel("C:\\project\\political_ner\\Insta_TikTok_AccountNames\\NER_Identify\\AccountName_Identified\\Sweden_AccName.xlsx", 
                index=False)

In [ ]:
df_ner = pd.read_excel("C:\\project\\political_ner\\Insta_TikTok_AccountNames\\NER_Identify\\AccountName_Identified\\Sweden_AccName.xlsx")

df_expanded = df_ner.assign(NER=df_ner['NER'].str.split(',')).explode('NER')
df_expanded['NER'] = df_expanded['NER'].str.strip()
df_expanded.reset_index(drop=True, inplace=True)
df_expanded

#ner_df_path = r'C:\project\political_ner\Main_NER\Finland_NER(3).xlsx'
df2_path = r'C:\project\political_ner\mep_party_cleaned3 (1) (1).xlsx'
df_party_path = r'C:\project\political_ner\party_names_merged 4(1) (2).xlsx'


df_ner = df_expanded
df2 = pd.read_excel(df2_path)
df_party = pd.read_excel(df_party_path)

df_ner['Country'] = 'Sweden'

additional_candidates = [
     {'Famname': 'Trump', 'Candidate': 'Donald Trump', 'Party': 'Republican Party', 'Country': 'United States'},
    {'Famname': 'Biden', 'Candidate': 'Joe Biden', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Hillary', 'Candidate': 'Hillary Clinton', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Clinton', 'Candidate': 'Hillary Clinton', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Obama', 'Candidate': 'Barack Obama', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Volodymyr', 'Candidate': 'Volodymyr Zelenskyy', 'Party': 'Servant of the People', 'Country': 'Ukraine'},
    {'Famname': 'Zelensky', 'Candidate': 'Volodymyr Zelenskyy', 'Party': 'Servant of the People', 'Country': 'Ukraine'},
    {'Famname': 'Putin', 'Candidate': 'Vladimir Putin', 'Party': 'United Russia', 'Country': 'Russia'},
    {'Famname': 'Harris', 'Candidate': 'Kamala Harris', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Kamala', 'Candidate': 'Kamala Harris', 'Party': 'Democratic Party', 'Country': 'United States'},
    {'Famname': 'Rishi', 'Candidate': 'Rishi Sunak', 'Party': 'Conservative Party', 'Country': 'United Kingdom'},
    {'Famname': 'Sunak', 'Candidate': 'Rishi Sunak', 'Party': 'Conservative Party', 'Country': 'United Kingdom'},
    {'Famname': 'Macron', 'Candidate': 'Emmanuel Macron', 'Party': 'La République En Marche!', 'Country': 'France'},
    {'Famname': 'Emmanuel', 'Candidate': 'Emmanuel Macron', 'Party': 'La République En Marche!', 'Country': 'France'},
    {'Famname': 'Scholz', 'Candidate': 'Olaf Scholz', 'Party': 'Social Democratic Party', 'Country': 'Germany'},
    {'Famname': 'Olaf', 'Candidate': 'Olaf Scholz', 'Party': 'Social Democratic Party', 'Country': 'Germany'},
    {'Famname': 'Jinping', 'Candidate': 'Xi Jinping', 'Party': 'Communist Party of China', 'Country': 'China'},
    {'Famname': 'Xi', 'Candidate': 'Xi Jinping', 'Party': 'Communist Party of China', 'Country': 'China'},
    {'Famname': 'Kim', 'Candidate': 'Kim Jong-un', 'Party': "Workers' Party of Korea", 'Country': 'North Korea'},
    {'Famname': 'Jong-un', 'Candidate': 'Kim Jong-un', 'Party': "Workers' Party of Korea", 'Country': 'North Korea'},    
    {'Famname': 'Orbán', 'Candidate': 'Viktor Orbán', 'Party': "Fidesz", 'Country': 'Hungary'},
    {'Famname': 'Orban', 'Candidate': 'Viktor Orban', 'Party': "Fidesz", 'Country': 'Hungary'},
    {'Famname': 'Órban', 'Candidate': 'Viktor Órban', 'Party': "Fidesz", 'Country': 'Hungary'},
    {'Famname': 'Schröder', 'Candidate': 'Gerhard Schröder', 'Party': "Social Democratic Party of Germany", 'Country': 'Germany'},
    {'Famname': 'Schroeder', 'Candidate': 'Gerhard Schröder', 'Party': "Social Democratic Party of Germany", 'Country': 'Germany'},
    

    

    
    # For Poland only - to be changed per country
    {'Famname': 'Jarosław', 'Candidate': 'Jarosław Kaczyński', 'Party': "Law and Justice", 'Country': 'Poland'},
    {'Famname': 'Jaroslaw', 'Candidate': 'Jarosław Kaczyński', 'Party': "Law and Justice", 'Country': 'Poland'},
    {'Famname': 'Kaczyński', 'Candidate': 'Jarosław Kaczyński', 'Party': "Law and Justice", 'Country': 'Poland'},
    {'Famname': 'Kaczynski', 'Candidate': 'Jarosław Kaczynski', 'Party': "Law and Justice", 'Country': 'Poland'},
    {"Famname":"Hołownia","Candidate":"Szymon Hołownia","Party":"Polska 2050","Country":"Poland"},
    {"Famname":"Holownia","Candidate":"Szymon Holownia","Party":"Polska 2050","Country":"Poland"},
    {"Famname":"Przemysław","Candidate":"Przemysław Czarnek","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Przemyslaw","Candidate":"Przemysław Czarnek","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Gembicka","Candidate":"Anna Gembicka","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Czarnek","Candidate":"Przemysław Czarnek","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Czarnek","Candidate":"Przemyslaw Czarnek","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Schetyna","Candidate":"Grzegorz Schetyna","Party":"Platforma Obywatelska","Country":"Poland"},
    {"Famname":"Grzegorz","Candidate":"Grzegorz Schetyna","Party":"Platforma Obywatelska","Country":"Poland"},
    {"Famname":"Trzaskowski","Candidate":"Rafał Trzaskowski","Party":"Platforma Obywatelska","Country":"Poland"},
    {"Famname":"Trzaskowski","Candidate":"Rafal Trzaskowski","Party":"Platforma Obywatelska","Country":"Poland"},
    {"Famname":"Berkowicz","Candidate":"Konrad Berkowicz","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Berkowicz","Candidate":"Konrad Berkowicz","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Zajączkowska-Hernik","Candidate":"Ewa Zajączkowska-Hernik","Party":"Nowa Nadzieja","Country":"Poland"},
    {"Famname":"Zajaczkowska-Hernik","Candidate":"Ewa Zajaczkowska-Hernik","Party":"Nowa Nadzieja","Country":"Poland"},
    {"Famname":"Zajaczkowska","Candidate":"Ewa Zajaczkowska-Hernik","Party":"Nowa Nadzieja","Country":"Poland"},
    {"Famname":"Hernik","Candidate":"Ewa Zajaczkowska-Hernik","Party":"Nowa Nadzieja","Country":"Poland"},
    {"Famname":"Macierewicz","Candidate":"Antoni Macierewicz","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Gosek","Candidate":"Mariusz Gosek","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Kowalski ","Candidate":"Janusz Kowalski","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Krystian","Candidate":"Mariusz Krystian","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Karozarowski","Candidate":"Piotr Korczarowski","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Korczarowski","Candidate":"Piotr Korczarowski","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Walker","Candidate":"Walker Justyna","Party":"Normal Country","Country":"Poland"},
    {"Famname":"Fritz","Candidate":"Roman Fritz","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Krasowicz","Candidate":"Krasowicz Justyna","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Morawiecka","Candidate":"Mateusz Morawiecki","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Morawiecki","Candidate":"Mateusz Morawiecki","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Karnowski","Candidate":"Karnowski Jacek Krzysztof","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"Morawiecki","Candidate":"Mateusz Morawiecki","Party":"Prawo i Sprawiedliwość","Country":"Poland"},
    {"Famname":"Zieloni/KO","Candidate":"Civic Coalition","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"Zieloni","Candidate":"Civic Coalition","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"KO","Candidate":"Civic Coalition","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"Kołodziejczak","Candidate":"Michał Kołodziejczak","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"Kolodziejczak","Candidate":"Michał Kołodziejczak","Party":"Civic Coalition","Country":"Poland"},
    {"Famname":"Mentzeń","Candidate":"Sławomir Mentzen","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Mentzen","Candidate":"Sławomir Mentzen","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Pelczynska-Nalecz","Candidate":"Katarzyna Pelczynska-Nalecz","Party":"Third Way","Country":"Poland"},
    {"Famname":"D.Tusk","Candidate":"Donald Tusk","Party":"Civic Platform Party","Country":"Poland"},
    {"Famname":"K. Bosak","Candidate":"Krzysztof Bosak","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"S. Mentzen","Candidate":"Sławomir Mentzen","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Razem/Lewica","Candidate":"The Left","Party":"The Left","Country":"Poland"},
    {"Famname":"D. Sośnierz","Candidate":"Dobromir Sośnierz","Party":"Confederation Liberty and Independence","Country":"Poland"},
    {"Famname":"Dana Jazlowiecka","Candidate":"Danuta Jazłowiecka","Party":"Civic Coalition","Country":"Poland"},
    
    
    


    # For Finland only - to be changed per country
    {'Famname': 'Purra', 'Candidate': 'Riikka Purra', 'Party': "Finns Party", 'Country': 'Finland'},
    {"Famname": "Ben Z", "Candidate": "Ben Zyskowicz", "Party": "National Coalition Party", "Country": "Finland"},
    {"Famname": "Bompardb", "Candidate": "Manuel Bompard", "Party": "La France Insoumise", "Country": "France"},
    {"Famname": "Bompart", "Candidate": "Manuel Bompard", "Party": "La France Insoumise", "Country": "France"},
    {"Famname": "Bordella", "Candidate": "Jordan Bardella", "Party": "National Rally", "Country": "France"},
    {"Famname": "Caro Ra", "Candidate": "Carola Rackete", "Party": "The Left (Ind)", "Country": "Germany"},
    {"Famname": "col. markov", "Candidate": "Nikolay Markov", "Party": "Velichie", "Country": "Bulgaria"},
    {"Famname": "collonel Markov", "Candidate": "Nikolay Markov", "Party": "Velichie", "Country": "Bulgaria"},
    {"Famname": "Davambazki", "Candidate": "Angel Dzhambazki", "Party": "IMRO – Bulgarian National Movement", "Country": "Bulgaria"},
    {"Famname": "Djambazki", "Candidate": "Angel Dzhambazki", "Party": "IMRO – Bulgarian National Movement", "Country": "Bulgaria"},
    {"Famname": "Dude", "Candidate": "Andrzej Duda", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Dzhambaski", "Candidate": "Angel Dzhambazki", "Party": "IMRO – Bulgarian National Movement", "Country": "Bulgaria"},
    {"Famname": "errejon", "Candidate": "Íñigo Errejón", "Party": "Sumar", "Country": "Spain"},
    {"Famname": "Fred. Merz", "Candidate": "Friedrich Merz", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "G. Braun", "Candidate": "Grzegorz Braun", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "G.Braun", "Candidate": "Grzegorz Braun", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Gasiuk-Pichowicz", "Candidate": "Kamila Gasiuk-Pihowicz", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Gluksman", "Candidate": "Raphaël Glucksmann", "Party": "Place Publique", "Country": "France"},
    {"Famname": "Gluksmann", "Candidate": "Raphaël Glucksmann", "Party": "Place Publique", "Country": "France"},
    {"Famname": "Haback", "Candidate": "Robert Habeck", "Party": "Alliance 90/The Greens", "Country": "Germany"},
    {"Famname": "Halbeck", "Candidate": "Robert Habeck", "Party": "Alliance 90/The Greens", "Country": "Germany"},
    {"Famname": "Hr.Ivanov", "Candidate": "Hristo Ivanov", "Party": "Yes, Bulgaria!", "Country": "Bulgaria"},
    {"Famname": "JL Abalos", "Candidate": "José Luis Ábalos", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "JL Albares", "Candidate": "José Manuel Albares Bueno", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "jm albares", "Candidate": "José Manuel Albares Bueno", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "jmalbares", "Candidate": "José Manuel Albares Bueno", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "Jonas S", "Candidate": "Jonas Sjöstedt", "Party": "The Left Party", "Country": "Sweden"},
    {"Famname": "jorge pueyo", "Candidate": "Jorge Pueyo", "Party": "Sumar", "Country": "Spain"},
    {"Famname": "jorge_pueyo", "Candidate": "Jorge Pueyo", "Party": "Sumar", "Country": "Spain"},
    {"Famname": "Makron", "Candidate": "Emmanuel Macron", "Party": "Renaissance", "Country": "France"},
    {"Famname": "Max Krah", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Max. Krahl", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Maxim. Krah", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Miljöp", "Candidate": "Miljöpartiet de Gröna", "Party": "The Green Party", "Country": "Sweden"},
    {"Famname": "MJ montero", "Candidate": "María Jesús Montero", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "MJMontero", "Candidate": "María Jesús Montero", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "Sawick", "Candidate": "Marek Sawicki", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Sawicki", "Candidate": "Marek Sawicki", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Torock", "Candidate": "László Toroczkai", "Party": "Our Homeland Movement", "Country": "Hungary"},
    {"Famname": "torockai", "Candidate": "László Toroczkai", "Party": "Our Homeland Movement", "Country": "Hungary"},
    {"Famname": "Torockay", "Candidate": "László Toroczkai", "Party": "Our Homeland Movement", "Country": "Hungary"},


    {"Famname": "Ulf Krist", "Candidate": "Ulf Kristersson", "Party": "The Moderate Party", "Country": "Sweden"},
    {"Famname": "Urs", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Ursula Van Leyden", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Ursula vdL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Vanya.gri", "Candidate": "Vanya Grigorova", "Party": "Solidary Bulgaria", "Country": "Bulgaria"},
    {"Famname": "VDL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Vucic", "Candidate": "Aleksandar Vučić", "Party": "Serbian Progressive Party", "Country": "Serbia"},
    {"Famname": "W. Buda", "Candidate": "Waldemar Buda", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "atteh", "Candidate": "Atte Harjanne", "Party": "Green League", "Country": "Finland"},
    {"Famname": "Höck", "Candidate": "Björn Höcke", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Krass", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Max Krahl", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Max. Krah", "Candidate": "Maximilian Krah", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Wiedel", "Candidate": "Alice Weidel", "Party": "Alternative for Germany", "Country": "Germany"},
    {"Famname": "Gutzanov", "Candidate": "Borislav Gutsanov", "Party": "Bulgarian Socialist Party", "Country": "Bulgaria"},
    {"Famname": "Lintila", "Candidate": "Mika Lintilä", "Party": "Centre Party", "Country": "Finland"},
    {"Famname": "Paakki", "Candidate": "Valtteri Paakki", "Party": "Centre Party", "Country": "Finland"},
    {"Famname": "DL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "U VDL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "U. VdL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "U. vdL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Urs.", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Ursula VDL", "Candidate": "Ursula von der Leyen", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "P. Armthor", "Candidate": "Philipp Amthor", "Party": "Christian Democratic Union", "Country": "Germany"},
    {"Famname": "Veber", "Candidate": "Manfred Weber", "Party": "Christian Social Union", "Country": "Germany"},
    {"Famname": "Soeder", "Candidate": "Markus Söder", "Party": "Christian Social Union in Bavaria", "Country": "Germany"},
    {"Famname": "Bodnar", "Candidate": "Adam Bodnar", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "D. Jonski", "Candidate": "Dariusz Joński", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Grodzki", "Candidate": "Tomasz Grodzki", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "J. Grabiec", "Candidate": "Jan Grabiec", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Józefaciuk", "Candidate": "Marcin Józefaciuk", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Kierwińki", "Candidate": "Marcin Kierwiński", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "M.Kierwinski", "Candidate": "Marcin Kierwiński", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Miszalski", "Candidate": "Aleksander Miszalski", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Sylwia Bielawska", "Candidate": "Sylwia Bielawska", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "T. Siemioniak", "Candidate": "Tomasz Siemoniak", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Trzkawski", "Candidate": "Rafał Trzaskowski", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "W. Zembaczyński", "Candidate": "Witold Zembaczyński", "Party": "Civic Platform", "Country": "Poland"},
    {"Famname": "Wielichowska", "Candidate": "Monika Wielichowska", "Party": "Civic Platform", "Country": "Poland"},

    {"Famname": "Brezhnev", "Candidate": "Leonid Ilyich Brezhnev", "Party": "Communist Party of the Soviet Union", "Country": "Soviet Union"},
    {"Famname": "Grabarczyk", "Candidate": "Tomasz Grabarczyk", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Jakubiak", "Candidate": "Marek Jakubiak", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Janusz Korwin-Mikke", "Candidate": "Janusz Korwin-Mikke", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "K. Ber", "Candidate": "Konrad Berkowicz", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Korwin-Mikke", "Candidate": "Janusz Korwin-Mikke", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Mikke", "Candidate": "Janusz Korwin-Mikke", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Płaczek-Kobiór", "Candidate": "Grzegorz Płaczek", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Wawer", "Candidate": "Michał Wawer", "Party": "Confederation Liberty and Independence", "Country": "Poland"},
    {"Famname": "Pressman", "Candidate": "David Pressman", "Party": "Democratic Party", "Country": "United States"},
    {"Famname": "MZP", "Candidate": "Péter Márki-Zay", "Party": "Everybody's Hungary People's Party", "Country": "Hungary"},
    {"Famname": "JL Melenchon", "Candidate": "Jean-Luc Mélenchon", "Party": "France Unbowed", "Country": "France"},
    {"Famname": "Melenchon", "Candidate": "Jean-Luc Mélenchon", "Party": "France Unbowed", "Country": "France"},
    {"Famname": "Ruffin", "Candidate": "François Ruffin", "Party": "France Unbowed", "Country": "France"},
    {"Famname": "Frau Zimmerman", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "M-A Zimmerman", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "Marco Buschmann", "Candidate": "Marco Buschmann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "Marie S-Z", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "Strack-Zimmerman", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "STreck-Zimmerman", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "Zimmerman", "Candidate": "Marie-Agnes Strack-Zimmermann", "Party": "Free Democratic Party", "Country": "Germany"},
    {"Famname": "Deffontaines", "Candidate": "Léon Deffontaines", "Party": "French Communist Party", "Country": "France"},
    {"Famname": "SHuff", "Candidate": "Shawn Huff", "Party": "Green League", "Country": "Finland"},

    {"Famname": "Erdogan", "Candidate": "Recep Tayyip Erdoğan", "Party": "Justice and Development Party", "Country": "Turkey"},
    {"Famname": "Baluch", "Candidate": "Anna Baluch", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Bortniczuk", "Candidate": "Kamil Bortniczuk", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Czaputowicz", "Candidate": "Jacek Czaputowicz", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Fogiel", "Candidate": "Radosław Fogiel", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Hamerski", "Candidate": "Jan Hamerski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Horała", "Candidate": "Marcin Horała", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "J. Borowiak", "Candidate": "Joanna Borowiak", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "J. Saryusz-Wolski", "Candidate": "Jacek Saryusz-Wolski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Jabłoński", "Candidate": "Paweł Jabłoński", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Kaleta", "Candidate": "Sebastian Kaleta", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Karczewski", "Candidate": "Stanisław Karczewski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Karski", "Candidate": "Karol Karski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Kolarski", "Candidate": "Wojciech Kolarski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "P. Jaki", "Candidate": "Patryk Jaki", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "R. Czarnecki", "Candidate": "Ryszard Czarnecki", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "S. Kaleta", "Candidate": "Sebastian Kaleta", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Sasin", "Candidate": "Jacek Sasin", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Suski", "Candidate": "Marek Suski", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Tarczyńska", "Candidate": "Dominik Tarczyński", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Zbigniew Rau", "Candidate": "Zbigniew Rau", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "Ziobro", "Candidate": "Zbigniew Ziobro", "Party": "Law and Justice", "Country": "Poland"},
    {"Famname": "P. Kowal", "Candidate": "Piotr Kowal", "Party": "Left", "Country": "Poland"},
    {"Famname": "Zukowska", "Candidate": "Anna Maria Żukowska", "Party": "Left", "Country": "Poland"},
    {"Famname": "Kuivalainen", "Candidate": "Maija Kuivalainen", "Party": "Left Alliance", "Country": "Finland"},
    {"Famname": "Cristina Mortágua", "Candidate": "Cristina Mortágua", "Party": "Left Block", "Country": "Portugal"},
    {"Famname": "Maria Escaja", "Candidate": "Maria Escaja", "Party": "Left Block", "Country": "Portugal"},
    {"Famname": "Zanni-ID", "Candidate": "Marco Zanni", "Party": "Lega", "Country": "Italy"},

    {"Famname": "Pevski", "Candidate": "Delyan Peevski", "Party": "Movement for Rights and Freedoms", "Country": "Bulgaria"},
    {"Famname": "Ben Z.", "Candidate": "Ben Zyskowicz", "Party": "National Coalition Party", "Country": "Finland"},
    {"Famname": "jockaman", "Candidate": "Jocka Träskbäck", "Party": "National Coalition Party", "Country": "Finland"},
    {"Famname": "Miala", "Candidate": "Maria Miala", "Party": "National Coalition Party", "Country": "Finland"},
    {"Famname": "Mazzola", "Candidate": "Roberta Metsola", "Party": "Nationalist Party", "Country": "Malta"},
    {"Famname": "Olszewski", "Candidate": "Marek Olszewski", "Party": "Nonpartisan Local Government Activists", "Country": "Poland"},
    {"Famname": "Almeida", "Candidate": "José Luis Martínez-Almeida", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Azcon", "Candidate": "Jorge Azcón", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "D. Ayuso", "Candidate": "Isabel Díaz Ayuso", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Diaz Ayuso", "Candidate": "Isabel Natividad Díaz Ayuso", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "diaz ayuso", "Candidate": "Isabel Natividad Díaz Ayuso", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "eduardo zaplana", "Candidate": "Eduardo Zaplana", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Feiijóo", "Candidate": "Alberto Núñez Feijóo", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "feijoo", "Candidate": "Alberto Núñez Feijóo", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "JL Martínez Almeida", "Candidate": "José Luis Martínez-Almeida", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "JM Margallo", "Candidate": "José Manuel García-Margallo", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Jose Maria Aznar", "Candidate": "José María Aznar", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Juan Diego Requena", "Candidate": "Juan Diego Requena", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Manuel Bautista", "Candidate": "Manuel Bautista", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "mjcatala", "Candidate": "María José Catalá", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Nuria Montes", "Candidate": "Nuria Montes de Diego", "Party": "People's Party", "Country": "Spain"},
    {"Famname": "Emilio Delgado", "Candidate": "Emilio Delgado", "Party": "Podemos", "Country": "Spain"},
    {"Famname": "i_montero", "Candidate": "Irene Montero", "Party": "Podemos", "Country": "Spain"},
    {"Famname": "i_montero_", "Candidate": "Irene Montero", "Party": "Podemos", "Country": "Spain"},
    {"Famname": "ionebelarra", "Candidate": "Ione Belarra Urteaga", "Party": "Podemos", "Country": "Spain"},
    {"Famname": "pablo_echenique_", "Candidate": "Pablo Echenique", "Party": "Podemos", "Country": "Spain"},
    {"Famname": "pablobustinduy", "Candidate": "Pablo Bustinduy", "Party": "Podemos", "Country": "Spain"},

    {"Famname": "Caputová", "Candidate": "Zuzana Čaputová", "Party": "Progressive Slovakia", "Country": "Slovakia"},
    {"Famname": "Zemoour", "Candidate": "Éric Zemmour", "Party": "Reconquest", "Country": "France"},
    {"Famname": "G Rufián", "Candidate": "Juan Gabriel Rufián Romero", "Party": "Republican Left of Catalonia", "Country": "Spain"},
    {"Famname": "G. Rufián", "Candidate": "Juan Gabriel Rufián Romero", "Party": "Republican Left of Catalonia", "Country": "Spain"},
    {"Famname": "Rufian", "Candidate": "Juan Gabriel Rufián Romero", "Party": "Republican Left of Catalonia", "Country": "Spain"},
    {"Famname": "G. Bush", "Candidate": "George Bush", "Party": "Republican Party", "Country": "United States"},
    {"Famname": "GW Bush", "Candidate": "George W. Bush", "Party": "Republican Party", "Country": "United States"},
    {"Famname": "K. Kostadionov", "Candidate": "Kostadin Kostadinov", "Party": "Revival", "Country": "Bulgaria"},
    {"Famname": "SW", "Candidate": "Sahra Wagenknecht", "Party": "Sahra Wagenknecht Alliance", "Country": "Germany"},
    {"Famname": "Wagenecht", "Candidate": "Sahra Wagenknecht", "Party": "Sahra Wagenknecht Alliance", "Country": "Germany"},
    {"Famname": "Kuehnert", "Candidate": "Kevin Kühnert", "Party": "Social Democratic Party", "Country": "Germany"},
    {"Famname": "Mesarosch", "Candidate": "Robin Mesarosch", "Party": "Social Democratic Party", "Country": "Germany"},
    {"Famname": "Miersch", "Candidate": "Matthias Miersch", "Party": "Social Democratic Party", "Country": "Germany"},
    {"Famname": "Steinmeir", "Candidate": "Frank-Walter Steinmeier", "Party": "Social Democratic Party", "Country": "Germany"},
    {"Famname": "Feijoo", "Candidate": "Alberto Núñez Feijóo", "Party": "Spanish People's Party", "Country": "Spain"},
    {"Famname": "JM Albares", "Candidate": "José Manuel Albares Bueno", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "marotoreyes", "Candidate": "Reyes Maroto", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "MJ Montero", "Candidate": "María Jesús Montero", "Party": "Spanish Socialist Workers' Party", "Country": "Spain"},
    {"Famname": "carlesesteveaparicio", "Candidate": "Carlos Esteve Aparicio", "Party": "SUMAR", "Country": "Spain"},
    {"Famname": "Errejon", "Candidate": "Íñigo Errejón", "Party": "SUMAR", "Country": "Spain"},

    {"Famname": "Lisnard", "Candidate": "David Lisnard", "Party": "The Republicans", "Country": "France"},
    {"Famname": "K. Perczyńska Nałęcz", "Candidate": "Katarzyna Pełczyńska-Nałęcz", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Kamysz-Kosiniak", "Candidate": "Władysław Kosiniak-Kamysz", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Misiek Kaminski", "Candidate": "Michał Kamiński", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Mucha", "Candidate": "Joanna Mucha", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "P. Hennig-Kloska", "Candidate": "Paulina Hennig-Kloska", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "P. Zgorzelski", "Candidate": "Piotr Zgorzelski", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Paszyk", "Candidate": "Krzysztof Paszyk", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "R. Petru", "Candidate": "Ryszard Petru", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Roza Thun", "Candidate": "Róża Thun", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Zalewski", "Candidate": "Paweł Zalewski", "Party": "Third Way", "Country": "Poland"},
    {"Famname": "Kulja Viktor", "Candidate": "Kulja András", "Party": "TISZA Party", "Country": "Hungary"},
    {"Famname": "Nagy Ervin", "Candidate": "Nagy Ervin", "Party": "TISZA Party", "Country": "Hungary"},
    {"Famname": "Puigdemonnt", "Candidate": "Carles Puigdemont", "Party": "Together for Catalonia", "Country": "Spain"},

    {"Famname": "Terapeutti Ville", "Candidate": "Ville Merinen", "Party": "Others", "Country": "Finland"},
    {"Famname": "Nanna Väätäinen", "Candidate": "Nanna Väätäinen", "Party": "Perussuomalaiset", "Country": "Finland"},

    {"Famname": "Sara Seppänen", "Candidate": "Sara Seppänen", "Party": "Perussuomalaiset", "Country": "Finland"},
    {"Famname": "Netanjahu", "Candidate": "Benjamin Netanyahu", "Party": "Likud", "Country": "Israel"},
    {"Famname": "Miko Bergbom", "Candidate": "Miko Bergbom", "Party": "Perussuomalaiset", "Country": "Finland"},
    {"Famname": "Alex Stubb", "Candidate": "Alexander Stubb", "Party": "National Coalition Party", "Country": "Finland"},

    {"Famname": "Boyko Borissov", "Candidate": "Boyko Borissov", "Party": "GERB", "Country": "Bulgaria"},
    {"Famname": "Borissov", "Candidate": "Boyko Borissov", "Party": "GERB", "Country": "Bulgaria"},
    {"Famname": "Assen Vassilev", "Candidate": "Asen Vaskov Vasilev", "Party": "We Continue The Change", "Country": "Bulgaria"},
    {"Famname": "Tynkkynen", "Candidate": "Sebastian Tynkkynen", "Party": "Finns Party", "Country": "Finland"}
    

]

additional_df = pd.DataFrame(additional_candidates)
df2 = pd.concat([df2, additional_df], ignore_index=True)

df_ner.fillna('', inplace=True)
df2['Candidate'] = df2['Candidate'].fillna('')
df2['Party'] = df2['Party'].fillna('')
df2['Country'] = df2['Country'].fillna('')
df2['Famname'] = df2['Famname'].fillna('')
df_party['Party_English'] = df_party['Party_English'].fillna('')
df_party['Paty_OwnLanguage'] = df_party['Paty_OwnLanguage'].fillna('')
df_party['Abbreviation'] = df_party['Abbreviation'].fillna('')
df_party['country_names'] = df_party['country_names'].fillna('')

# Standardize country names
df2['Country'] = df2['Country'].str.lower().str.strip()
df_party['country_names'] = df_party['country_names'].str.lower().str.strip()
df_ner['Country'] = df_ner['Country'].str.lower().str.strip()

df_ner['Names'] = ''
df_ner['Party'] = ''

stop_words = set(stopwords.words('english'))
punctuation_table = str.maketrans('', '', string.punctuation)

def preprocess_ner_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower().strip()
    text = text.translate(punctuation_table)
    return text

df2['phrase_list'] = df2.apply(
    lambda x: [preprocess_ner_text(x['Candidate'])] +
    ([preprocess_ner_text(x['Famname'])] if x['Famname'] else []),
    axis=1
)

candidate_phrase_dict = {}
for _, row in df2.iterrows():
    for phrase in row['phrase_list']:
        candidate_phrase_dict[phrase] = (row['Candidate'], row['Party'], row['Country'])

def process_ner_row(args):
    index, row = args
    try:
        return func_timeout(120, process_ner_row_inner, args=(index, row))
    except FunctionTimedOut:
        print(f"Row {index} processing timed out.")
        return (index, None, None)
    except Exception as e:
        print(f"Error processing row {index}: {e}")
        return (index, None, None)

def process_ner_row_inner(index, row):
    ner_text = row['NER']
    if not isinstance(ner_text, str):
        ner_text = str(ner_text)

    ner_text_processed = preprocess_ner_text(ner_text)
    match_name, match_party = None, None

    best_score = 0
    for candidate_phrase, (candidate_name, party, _) in candidate_phrase_dict.items():
        score = fuzz.token_set_ratio(ner_text_processed, candidate_phrase)
        if score >= 82 and score > best_score:
            best_score = score
            match_name, match_party = candidate_name, party

    return (index, match_name, match_party)

# Process each row in df_ner
results = []

for index, row in tqdm(df_ner.iterrows(), total=len(df_ner)):
    result = process_ner_row((index, row))
    results.append(result)

for result in results:
    index, name, party = result
    if name:
        df_ner.at[index, 'Names'] = name
        df_ner.at[index, 'Party'] = party

# Create party_phrase_dict that maps phrases to lists of parties with their countries
df_party['phrase_list'] = df_party.apply(
    lambda x: [preprocess_ner_text(x['Party_English'])] +
    ([preprocess_ner_text(x['Paty_OwnLanguage'])] if x['Paty_OwnLanguage'] else []) +
    ([preprocess_ner_text(x['Abbreviation'])] if x['Abbreviation'] else []),
    axis=1
)

party_phrase_dict = {}
for _, row in df_party.iterrows():
    for phrase in row['phrase_list']:
        if phrase not in party_phrase_dict:
            party_phrase_dict[phrase] = []
        party_phrase_dict[phrase].append((row['Party_English'], row['country_names']))

def process_party_row(row):
    ner_text = row['NER']
    if not isinstance(ner_text, str):
        ner_text = str(ner_text)
    ner_text_processed = preprocess_ner_text(ner_text)
    current_party = row['Party']
    row_country = row['Country'].lower()
    best_score = 0
    match_party = current_party  # Default to existing party

    potential_matches = []

    # Collect potential matches
    for phrase, parties in party_phrase_dict.items():
        score = fuzz.token_set_ratio(ner_text_processed, phrase)
        if score >= 85:
            for party_name, country_name in parties:
                potential_matches.append((score, party_name, country_name.lower()))

    # Prioritize parties from the same country
    same_country_matches = [m for m in potential_matches if m[2] == row_country]
    if same_country_matches:
        best_match = max(same_country_matches, key=lambda x: x[0])
    elif not current_party and potential_matches:
        # Only consider other countries if 'Party' is empty
        best_match = max(potential_matches, key=lambda x: x[0])
    else:
        best_match = None

    if best_match:
        best_score = best_match[0]
        potential_match = best_match[1]

    # Apply the matching logic
    if best_score >= 90:
        match_party = potential_match
    elif best_score >= 85 and current_party:
        match_party = potential_match
    # Else, keep the existing 'Party' value

    return match_party

# Apply the updated process_party_row function
df_ner['Party'] = df_ner.apply(process_party_row, axis=1)




In [ ]:

# Save the updated DataFrame to an Excel file
output_path = r'C:\project\political_ner\Insta_TikTok_AccountNames\NER_Identify\AccountName_Identified\SE_AccName_Identified.xlsx'
df_ner.to_excel(output_path, index=False)
print(f"File saved to: {output_path}")

In [ ]:
import requests
from bs4 import BeautifulSoup

# Search keywords
search_term = "scrapingbypass"

# Request headers
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"}

# Search pages of results
num_pages = 5
no=1

for page in range(0,num_pages):
    # Request URL
    if page == 0:
        url = f"https://www.google.com/search?q={search_term}"
    else:
        url = f"https://www.google.com/search?q={search_term}&start={page*10}"

    # Request
    response = requests.get(url, headers=headers)

    # Parse HTML
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract search reult
    search_results = soup.select(".yuRUbf")

    # Print title and link
    for result in search_results:
        title = result.select_one("h3").text
        link = result.select_one("a")["href"]
        print(f"{no}: {title}: {link}")
        no=no+1

In [ ]:
# Here add the following:
# Add the greens party of that countyr manually 
# Add the country and if EU or not

In [ ]:
df3000 = pd.read_excel(r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_SE_NER_Identified.xlsx')

In [ ]:
import pandas as pd
import numpy as np
import json
from thefuzz import fuzz

df = df3000

def is_green_fuzzy(ner_value, threshold=90):
    """
    Returns True if ner_value either exactly matches 'Green'/'Greens'
    or if fuzzy similarity >= threshold to either of them.
    """
    if pd.isnull(ner_value):
        return False
    
    ner_lower = ner_value.lower()
    # Exact check
    if ner_lower in ['green', 'greens']:
        return True
    
    # Fuzzy check
    targets = ['green', 'greens']
    for t in targets:
        if fuzz.ratio(ner_lower, t) >= threshold:
            return True
    
    return False

mask = df['NER'].apply(is_green_fuzzy)

old_rows = df.loc[mask, ['Names', 'Party']].copy()

df.loc[mask, 'Names'] = np.nan
df.loc[mask, 'Party'] = "Miljöpartiet de Gröna"

new_rows = df.loc[mask, ['Names', 'Party']].copy()

changes = []
for idx in old_rows.index:
    changes.append({
        'index': int(idx),
        'old': {
            'Names': old_rows.at[idx, 'Names'],
            'Party': old_rows.at[idx, 'Party']
        },
        'new': {
            'Names': new_rows.at[idx, 'Names'],
            'Party': new_rows.at[idx, 'Party']
        }
    })

print(json.dumps(changes, indent=2, ensure_ascii=False))



In [ ]:
output_path = r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_HU_NER_Identified.xlsx'
df.to_excel(output_path, index=False)


In [ ]:
df4000 = pd.read_excel(r'C:\project\political_ner\mep_party_cleaned3 (1) (1).xlsx')
df4000

In [ ]:
import pandas as pd
import numpy as np
from thefuzz import fuzz

df = pd.read_excel(r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_SE_NER_Identified.xlsx')
df4000 = pd.read_excel(r'C:\project\political_ner\mep_party_cleaned3 (1) (1).xlsx')

def normalize_string(s: str) -> str:
    """
    Lowercases, strips extra whitespace, etc.
    """
    return ' '.join(s.lower().split())

def get_country_if_match(name: str, party: str, df_lookup: pd.DataFrame, threshold=95):
    """
    For a given (name, party), search df_lookup (Candidate, Party, Country)
    Return the matched 'Country' if both name & party match with ≥ threshold fuzzy ratio
    OR if they match exactly ignoring case/whitespace.
    If multiple matches exist, the first found is returned. Otherwise, returns np.nan.
    """

    norm_name = normalize_string(name) if pd.notnull(name) else ""
    norm_party = normalize_string(party) if pd.notnull(party) else ""

    best_country = np.nan

    for _, row in df_lookup.iterrows():
        candidate = normalize_string(str(row['Candidate']))
        candidate_party = normalize_string(str(row['Party']))
        candidate_country = row['Country']

        # Calculate name similarity
        # If exact ignoring case/whitespace => treat as 100
        if norm_name == candidate:
            name_ratio = 100
        else:
            name_ratio = fuzz.ratio(norm_name, candidate)

        # Calculate party similarity
        if norm_party == candidate_party:
            party_ratio = 100
        else:
            party_ratio = fuzz.ratio(norm_party, candidate_party)

        # Check if both are above threshold
        if name_ratio >= threshold and party_ratio >= threshold:
            best_country = candidate_country
            # You could break on the first match, or keep searching 
            # if you want the "best" among multiples. 
            # For simplicity, just break on first valid match.
            break

    return best_country

df['NER_country'] = df.apply(
    lambda row: get_country_if_match(row['Names'], row['Party'], df4000),
    axis=1
)

eu_countries = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Germany ", " Germany"," Germany ", "Greece", "Hungary",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden"
}

df['EU'] = df['NER_country'].apply(lambda c: 1 if c in eu_countries else 0)

df



In [ ]:
output_path = r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_SE_NER_Identified.xlsx'
df.to_excel(output_path, index=False)

In [ ]:
import pandas as pd
import numpy as np
from thefuzz import fuzz

df = pd.read_excel(r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_SE_NER_Identified.xlsx')
df4000 = pd.read_excel(r'C:\project\political_ner\mep_party_cleaned3 (1) (1).xlsx')
additional_df = additional_df

common_cols = ['Candidate','Party','Country']
df_lookup = pd.concat([
    df4000[common_cols],
    additional_df[common_cols]
], ignore_index=True)

def normalize_string(s):
    if pd.isnull(s):
        return ""
    return ' '.join(s.lower().split())

def get_country_if_match(name: str, party: str, df_ref: pd.DataFrame, threshold=95):
    """Return the matched 'Country' from df_ref if both name & party match 
       with ≥ threshold fuzzy ratio OR exact ignoring case/whitespace.
       If multiple matches, returns the first match found.
       Otherwise, returns np.nan.
    """
    norm_name = normalize_string(name)
    norm_party = normalize_string(party)

    # Loop over df_ref rows
    best_country = np.nan
    for _, row in df_ref.iterrows():
        candidate = normalize_string(str(row['Candidate']))
        candidate_party = normalize_string(str(row['Party']))
        candidate_country = row['Country']

        # Name similarity (check exact or fuzzy)
        if norm_name == candidate:
            name_ratio = 100
        else:
            name_ratio = fuzz.ratio(norm_name, candidate)

        # Party similarity
        if norm_party == candidate_party:
            party_ratio = 100
        else:
            party_ratio = fuzz.ratio(norm_party, candidate_party)

        # If both above threshold, we have a match
        if name_ratio >= threshold and party_ratio >= threshold:
            best_country = candidate_country
            break

    return best_country

# ------------------------------------------------------
# 7) Create the NER_country column
#    Using the combined lookup (df_lookup)
# ------------------------------------------------------
df['NER_country'] = df.apply(
    lambda row: get_country_if_match(row['Names'], row['Party'], df_lookup),
    axis=1
)

# ------------------------------------------------------
# 8) Fixing the EU column
#    a) Normalize country synonyms (like 'french' -> 'France')
#    b) Then check membership in the EU set
# ------------------------------------------------------

# (a) Let's create a small dictionary for synonyms or adjectives → official country name.
#     Feel free to add your own custom synonyms as needed.
country_synonyms = {
    'french': 'France',
    'german': 'Germany',
    'swedish': 'Sweden',
    'polish': 'Poland',
    'spanish': 'Spain',
    'finnish': 'Finland'
}

# EU membership set (country names in official form)
eu_countries = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden"
}

def normalize_country(country):
    """
    1) Normalize whitespace/case
    2) Map synonyms -> official country name
    """
    if pd.isnull(country):
        return None
    country_str = country.strip().lower()

    # If the normalized string is in synonyms dict, replace
    if country_str in country_synonyms:
        return country_synonyms[country_str]

    # Otherwise, we attempt to 'title' it except for known multi-word countries.
    # e.g., "united kingdom" -> "United Kingdom"
    # If your data is inconsistent, you might prefer a more robust approach.  
    # We'll do a simple .title() for demonstration
    return ' '.join(word.capitalize() for word in country_str.split())

# Apply the normalizer
df['NER_country_cleaned'] = df['NER_country'].apply(normalize_country)

# (b) Now build the EU column
def is_eu_member(country_cleaned):
    if not country_cleaned or pd.isnull(country_cleaned):
        return 0
    # The dictionary we've used capitalizes each word, so let's unify again
    # e.g., "United Kingdom" or "France".
    # We'll create a direct mapping:
    # The official set uses e.g. "France", "Germany", "United Kingdom" is *not* in the EU
    # So let's unify the final form to match exactly
    return 1 if country_cleaned in eu_countries else 0

df['EU'] = df['NER_country_cleaned'].apply(is_eu_member)

# ------------------------------------------------------
# 9) Check results
# ------------------------------------------------------
df

# ------------------------------------------------------
# 10) (Optional) Save to Excel
# ------------------------------------------------------
# df.to_excel("path_to_output.xlsx", index=False)


In [ ]:
df = df.drop(columns=['NER_country'])
df

In [ ]:
output_path = r'C:\project\political_ner\FINAL_NER_2025_17_01\Final_SE_NER_Identified.xlsx'
df.to_excel(output_path, index=False)